<a href="https://colab.research.google.com/github/Nasim-Amani/tag-recommender-agent-learning/blob/main/hands-on-levels/L2_Borrow_tags_from_neighbors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Level 2: Borrow tags from neighbors

**Goal:** a new product probably has the same tags as the old products it resembles. So: find its **5 most similar training products**, and recommend the tags **they** have.

**Steps (example: a new "Argan Oil Repair Shampoo")**
1. **Find neighbors:** compare the shampoo's text vector with every **training** product (same BERT vectors as Level 1) and keep the 5 most similar, e.g. other repair shampoos and a hair mask.
2. **Collect their tags:** look at the true tags of those 5 neighbors (allowed: they are training products).
3. **Vote:** each tag's score = how many neighbors have it.
   e.g. `Hair Type: dry` on 4 of 5 neighbors → 4 votes; `Scent: lavender` on 1 → 1 vote.
4. **Top 10:** the tags with the most votes.

**Two versions:**
- **Vote:** every neighbor counts the same.
- **Weighted:** closer neighbors count more (each vote × similarity).

**Difference from Level 1:** text similarity compares the product with tag **names**; neighbors compare the product with other **products** and copy their **real tags**. This is the **collaborative filtering** idea, made to work for new products by finding neighbors through text.

In [2]:
!pip install -q sentence-transformers

setup, the same as Level 1 (data, tags, split)

In [3]:
import json, gzip, os, urllib.request
from collections import Counter
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# 1) Download the two files (only the first time)
base = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
for url, name in [(f"{base}/meta_categories/meta_All_Beauty.jsonl.gz", "meta.jsonl.gz"),
                  (f"{base}/review_categories/All_Beauty.jsonl.gz", "reviews.jsonl.gz")]:
    if not os.path.exists(name):
        urllib.request.urlretrieve(url, name)

# 2) Read the first 2,000 products
products = []
for line in gzip.open("meta.jsonl.gz", "rt"):
    products.append(json.loads(line))
    if len(products) == 2000:
        break

# 3) Reviews and first-review time of those products
reviews = {p["parent_asin"]: [] for p in products}
first_time = {}
for line in gzip.open("reviews.jsonl.gz", "rt"):
    r = json.loads(line)
    a = r["parent_asin"]
    if a in reviews:
        reviews[a].append(r["text"])
        first_time[a] = min(first_time.get(a, r["timestamp"]), r["timestamp"])

# 4) Tags = values of seller attributes, minus identifiers and junk keys
SKIP = {"Brand", "Manufacturer", "UPC", "Product Dimensions", "Item model number",
        "Date First Available", "Item Weight", "Package Dimensions",
        "Is Discontinued By Manufacturer", "Unit Count", "Number of Items",
        "Age Range (Description)", "Mounting Type", "Installation Type",
        "Package Information", "Extension Length"}

def all_tags_of(p):
    tags = []
    for key, value in p["details"].items():
        if key in SKIP or not isinstance(value, str):
            continue
        for v in value.split(","):                 # "Oily, Dry" -> two tags
            v = v.strip().lower()
            if v and v != "all":                   # "suits everyone" is not a tag
                tags.append(f"{key}: {v}")
    return tags

counts = Counter(t for p in products for t in all_tags_of(p))
candidate_tags = sorted(t for t, c in counts.items() if c >= 20)   # tags on >= 20 products
products = [p for p in products if set(all_tags_of(p)) & set(candidate_tags)]
n, m = len(products), len(candidate_tags)

# 5) Answer key Y: Y[k, j] = 1 if product k has tag j
Y = np.array([[t in all_tags_of(p) for t in candidate_tags] for p in products], dtype=float)

# 6) Split by time: oldest 70% train, next 15% validation, newest 15% test (= new products)
t = np.array([first_time.get(p["parent_asin"], np.inf) for p in products])
order = np.argsort(t)
train_idx = order[: int(0.70 * n)]
val_idx   = order[int(0.70 * n): int(0.85 * n)]
test_idx  = order[int(0.85 * n):]
print(n, "products |", m, "tags |", len(train_idx), "train /", len(val_idx), "val /", len(test_idx), "test")

1057 products | 35 tags | 739 train / 159 val / 159 test


metric, BERT vectors and the Level 1 baselines (for comparison)

In [4]:
def evaluate(scores, products_idx, K=10):
    """scores: (products, tags), higher = more likely -> Precision@K, Recall@K, NDCG@K."""
    truth = Y[products_idx]
    top = np.argsort(-scores, axis=1)[:, :K]                    # the K best tags per product
    hits = np.take_along_axis(truth, top, axis=1)               # 1 where a recommended tag is correct
    n_true = truth.sum(axis=1)
    discount = 1 / np.log2(np.arange(2, K + 2))                 # position 1 counts most
    ideal = np.array([discount[:int(min(nt, K))].sum() for nt in n_true])
    return {"Precision@10": (hits.sum(axis=1) / K).mean(),
            "Recall@10":    (hits.sum(axis=1) / n_true).mean(),
            "NDCG@10":      ((hits * discount).sum(axis=1) / ideal).mean()}

# BERT vectors for products and tags
encoder = SentenceTransformer("all-MiniLM-L6-v2")
texts = [p["title"] + " " + " ".join(p["features"] + p["description"]) for p in products]
Z_prod = encoder.encode(texts, normalize_embeddings=True)          # (n, 384)
Z_tag  = encoder.encode(candidate_tags, normalize_embeddings=True) # (m, 384)

# Level 1 baselines
results = {}
rng = np.random.default_rng(0)
results["Random"] = evaluate(rng.random((len(test_idx), m)), test_idx)
results["Popularity"] = evaluate(np.tile(Y[train_idx].sum(axis=0), (len(test_idx), 1)), test_idx)
results["Text similarity"] = evaluate(Z_prod[test_idx] @ Z_tag.T, test_idx)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# ===== Level 2: borrow tags from the most similar TRAINING products =====
K_NEIGHBORS = 5

# 1) Similarity of every test product to every training product (BERT vectors from Level 1)
sims = Z_prod[test_idx] @ Z_prod[train_idx].T            # (test products, train products)

# 2) The 5 most similar training products for each test product
top = np.argsort(-sims, axis=1)[:, :K_NEIGHBORS]          # positions inside train_idx
neighbors = train_idx[top]                                # real product numbers, (test, 5)
neighbor_sims = np.take_along_axis(sims, top, axis=1)     # their similarity scores, (test, 5)

# 3a) Vote: each tag's score = number of neighbors that have it
#     Y[neighbors] is (test, 5, tags) -> sum over the 5 neighbors -> (test, tags)
votes = Y[neighbors].sum(axis=1)
results["Neighbors (vote)"] = evaluate(votes, test_idx)

# 3b) Weighted vote: closer neighbors count more (each neighbor's tags x its similarity)
weighted = (neighbor_sims[:, :, None] * Y[neighbors]).sum(axis=1)
results["Neighbors (weighted)"] = evaluate(weighted, test_idx)

table = pd.DataFrame(results).T.round(3)
table

,Precision@10,Recall@10,NDCG@10
Random,0.048,0.241,0.133
Popularity,0.116,0.573,0.340
Text similarity,0.123,0.651,0.484
Neighbors (vote),0.148,0.765,0.630
Neighbors (weighted),0.148,0.765,0.636



**Goal:** check whether borrowing tags from the 5 most similar training products beats the Level 1 methods on new products.

| Method | Precision@10 | Recall@10 | NDCG@10 |
|---|---|---|---|
| Text similarity (Level 1 best) | 0.123 | 0.651 | 0.484 |
| **Neighbors (vote)** | 0.148 | 0.765 | 0.630 |
| **Neighbors (weighted)** | **0.148** | **0.765** | **0.636** |

**What this means:**
- **Big jump:** NDCG goes from **0.48 → 0.64** (about **+31%**), and Recall from **65% → 77%** of true tags found. Copying the neighbors' real tags works much better than matching tag names.
- **Why it works better:** many tags are never written in the product text (e.g. `Item Form: cream` for a moisturizer). Text similarity can't see them; the neighbors **already have** them.
- **Weighted ≈ Vote (0.636 vs 0.630):** the 5 neighbors are about equally similar, so weighting them barely changes anything. This is the same effect as the near-equal 0.20 attention weights in the CASTRA paper.
- **Simple but strong:** no training at all, yet it clearly beats every Level 1 method.

**New bar for Level 3:** a method must beat **NDCG@10 = 0.636**.